<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/3_attractors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cinematic Strange Attractors

This notebook generates a 45-second high-definition 3D animation of three famous chaotic systems: the **Lorenz System**, the **Rössler Attractor**, and the **Thomas Cyclical Attractor**.

### Features
*   **Persistent Trails:** Each system features 150 particles with trails that remain visible over time.
*   **Cinematic Transitions:** 5-second overlapping cross-fades between systems.
*   **Dynamic Captions:** Titles for each attractor fade in at their peak visual climax.
*   **Descriptive UI:** Includes a running timestamp and orbiting camera path.
*   **Optimized Rendering:** Uses `Line3DCollection` for efficient CPU rendering on Google Colab.

### Attractors Visualized
1. **Lorenz:** Fire/Gold color theme.
2. **Rössler:** Teal/Violet color theme.
3. **Thomas:** Rose-gold color theme.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.colors import LinearSegmentedColormap
import cv2
from tqdm import tqdm
from IPython.display import HTML
from base64 import b64encode
import os

# --- Configuration ---
FPS = 30
DURATION = 45
WIDTH, HEIGHT = 720, 720
DPI = 100
NUM_PARTICLES = 150 # Balanced for speed and visual density
OVERLAP_SEC = 5
SEC_PER_SCENE = 15

def lorenz(pos, s=10, r=28, b=2.667): return np.array([s*(pos[1]-pos[0]), pos[0]*(r-pos[2])-pos[1], pos[0]*pos[1]-b*pos[2]])
def rossler(pos, a=0.2, b=0.2, c=5.7): return np.array([-pos[1]-pos[2], pos[0]+a*pos[1], b+pos[2]*(pos[0]-c)])
def thomas(pos, b=0.2081): return np.array([-b*pos[0] + np.sin(pos[1]), -b*pos[1] + np.sin(pos[2]), -b*pos[2] + np.sin(pos[0])])

scenes = [
    {'name': 'Lorenz System', 'func': lorenz, 'dt': 0.01, 'scale': 45, 'cmap': ['#ff0000', '#ff8800', '#ffff00']},
    {'name': 'Rössler Attractor', 'func': rossler, 'dt': 0.05, 'scale': 35, 'cmap': ['#00ffff', '#8a2be2', '#4b0082']},
    {'name': 'Thomas Cyclical', 'func': thomas, 'dt': 0.15, 'scale': 6, 'cmap': ['#ffafaf', '#e0bfb8', '#b76e79']}
]

def generate_cinematic_attractors():
    video_path = 'attractors_final.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(video_path, fourcc, FPS, (WIDTH, HEIGHT))
    fig = plt.figure(figsize=(WIDTH/DPI, HEIGHT/DPI), dpi=DPI, facecolor='black')
    ax = fig.add_subplot(111, projection='3d', facecolor='black')

    # Pre-calculate positions to handle overlaps smoothly
    all_pos = []
    for scene in scenes:
        p = np.random.uniform(-0.1, 0.1, (NUM_PARTICLES, 3))
        hist = [p.copy()]
        for _ in range(FPS * (SEC_PER_SCENE + OVERLAP_SEC)):
            p += scene['func'](p.T).T * scene['dt']
            hist.append(p.copy())
        all_pos.append(np.array(hist))

    for f in tqdm(range(FPS * DURATION), desc="Rendering Video"):
        ax.clear(); ax.set_axis_off(); ax.set_facecolor('black')
        current_time = f / FPS

        # Determine which scenes are active for overlapping
        active_scenes = []
        if current_time < 15: active_scenes.append(0)
        if 10 < current_time < 30: active_scenes.append(1)
        if 25 < current_time <= 45: active_scenes.append(2)

        for s_idx in active_scenes:
            scene = scenes[s_idx]
            # Calculate local frame for the scene data
            start_time = s_idx * (SEC_PER_SCENE - OVERLAP_SEC)
            local_f = int((current_time - start_time) * FPS)
            if local_f < 0 or local_f >= len(all_pos[s_idx]): continue

            # Opacity/Fade handling for transitions
            alpha_mod = 1.0
            if s_idx == 0 and current_time > 10: alpha_mod = (15 - current_time) / 5
            elif s_idx == 1:
                if current_time < 15: alpha_mod = (current_time - 10) / 5
                elif current_time > 25: alpha_mod = (30 - current_time) / 5
            elif s_idx == 2 and current_time < 30: alpha_mod = (current_time - 25) / 5

            my_cmap = LinearSegmentedColormap.from_list('c', scene['cmap'])
            colors = my_cmap(np.linspace(0, 1, NUM_PARTICLES))
            data = all_pos[s_idx][:local_f+1]

            for p in range(0, NUM_PARTICLES, 3):
                pts = data[:, p, :].reshape(-1, 1, 3)
                if len(pts) < 2: continue
                segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
                lc = Line3DCollection(segs, colors=colors[p], linewidths=0.6, alpha=0.4 * alpha_mod)
                ax.add_collection3d(lc)

            # Captions at climax (middle of scene duration)
            climax_time = start_time + (SEC_PER_SCENE / 2)
            if abs(current_time - climax_time) < 2:
                txt_alpha = 1.0 - abs(current_time - climax_time) / 2
                fig.text(0.5, 0.85, scene['name'], color='white', ha='center', fontsize=20, weight='bold', alpha=txt_alpha)

            # Scale camera per scene
            s = scene['scale']
            ax.set_xlim([-s, s]); ax.set_ylim([-s, s]); ax.set_zlim([-s, s])

        # Timestamp and orbiting camera
        ax.view_init(elev=20, azim=f * 0.4)
        fig.text(0.8, 0.05, f"T+ {current_time:.1f}s", color='gray', fontsize=10, family='monospace')

        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(HEIGHT, WIDTH, 4)
        video_writer.write(cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR))
        fig.texts.clear()

    video_writer.release(); plt.close(fig)
    mp4 = open(video_path,'rb').read()
    return HTML(f'<video width="{WIDTH}" height="{HEIGHT}" controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

generate_cinematic_attractors()

Rendering Video: 100%|██████████| 1350/1350 [21:57<00:00,  1.02it/s]
